# 04 · Resumen ejecutivo y roadmap hacia la v2

**Alcance:** consolidar la recomendación para la v1, y dejar **esbozados** (medio plazo) el
conjunto de consultas realista, la extensión del corpus a Competencias y una sección lista
para incorporar al LaTeX (**puntos 5, 6 y 7**).

In [1]:
import json
from pathlib import Path
RAIZ = Path.cwd();
if RAIZ.name == "notebooks": RAIZ = RAIZ.parent
res = json.loads((RAIZ/"experimentos"/"resultados"/"resultados.json").read_text(encoding="utf-8"))
import pandas as pd
pd.set_option("display.float_format", lambda x: f"{x:.3f}")
tab = pd.DataFrame({n: {"nDCG@10": v["ndcg@10"], "MRR@10": v["mrr@10"],
                        "Recall@100": v["recall@100"], "P@10": v["p@10"]}
                    for n, v in res["global"].items()}).T
print("Encoder:", res["encoder"], "| consultas con gold:", res["n_consultas_con_gold"])
tab.style.highlight_max(axis=0, color="#c9f0c9").format("{:.3f}")

Encoder: jina-embeddings-v3 | consultas con gold: 40


,nDCG@10,MRR@10,Recall@100,P@10
BM25 (referencia léxica),0.843,0.836,1.000,0.113
Denso puro,0.874,0.843,1.000,0.117
Denso + ontología,0.917,0.898,1.000,0.117
Híbrido (sin ontología),0.878,0.863,1.000,0.117
Híbrido + ontología,0.936,0.925,1.000,0.117
Híbrido + ontología + realce,0.911,0.894,1.000,0.117


## 1. Resumen ejecutivo

- **Problema.** Búsqueda semántica de reportes con énfasis en consultas por **nombre propio /
  entidad exacta**, donde la búsqueda léxica actual falla.
- **Qué se construyó.** Un framework de evaluación reproducible (protocolo *known-item*), una
  **escalera de ablación de 5 configuraciones** y benchmarks de eficiencia (Matryoshka,
  chunking), todo con el encoder real `jina-embeddings-v3`.
- **Hallazgo principal.** La mejor configuración es **Híbrido (BM25 + denso) + ontología**
  (nDCG@10 = 0.936 global; el máximo en el estrato de entidad exacta). La ontología aporta de
  forma consistente (P3); la fusión híbrida supera al denso y al léxico puros (P1, P2).
- **Realce por entidad (Config 5): descartado en v1.** Empeora por un detector poco
  discriminativo (notebook 02); queda como mejora candidata para v2 con un detector mejor.
- **Eficiencia.** La truncación Matryoshka reduce memoria del índice con retención de calidad
  cercana a 1.0 (notebook 03). El chunking no aporta con documentos cortos.

> **Configuración recomendada para v1:** Híbrido + ontología, embedding truncado a la menor
> dimensión con retención ≥ 0.99, índice denso por coseno exacto, **sin realce**.

**Limitación honesta.** El protocolo *known-item* con consultas derivadas de los documentos y
un corpus de 17 reportes hace que el **valor absoluto** de las métricas no sea precisión de
producción, y que Recall@100 sea trivialmente 1.0. Las conclusiones son **comparativas** entre
configuraciones. Cerrar esta brecha es el objetivo central de la v2.

## 2. (Punto 5) Conjunto de consultas realista — especificación y plan

**Objetivo.** Sustituir/complementar las consultas sintéticas *known-item* por consultas
**reales** con juicios de relevancia de expertos.

**Especificación propuesta.**
- **Fuentes de consultas reales:** registros/logs de búsqueda de la plataforma, tickets de
  soporte y entrevistas con usuarios (analistas, delegados). Meta inicial: **80–120 consultas**.
- **Estratos** (mantener trazabilidad con P1–P6): entidad_exacta, ambigua, documento_largo,
  competencias y un nuevo estrato **operacional** (consultas tal como las escribe el usuario,
  con errores y jerga).
- **Juicios de relevancia graduada** (0–3) por **≥ 2 evaluadores**, midiendo acuerdo
  (κ de Cohen / α de Krippendorff); resolver desacuerdos por adjudicación.
- **Pooling:** juzgar el top-k combinado de todas las configuraciones para no sesgar a favor de
  ninguna.

**Plan con expertos.**
1. Taller de 90 min para definir intenciones de búsqueda y ejemplos por estrato.
2. Recolección de consultas reales (anonimizadas, sin PII).
3. Rondas de anotación con guía de relevancia; medir acuerdo y calibrar.
4. Congelar `consultas_eval_v2.jsonl` + `qrels_v2.jsonl` con el mismo formato actual (el
   framework los admite sin cambios).

## 3. (Punto 6) Extensión del corpus a Competencias (v2)

**Motivación.** Las 11 consultas de *competencias* no tienen documento relevante en la v1
(**vacío de cobertura** medido, no un fallo del recuperador). Es la brecha de mayor impacto.

**Plan concreto.**
- **Ingesta:** incorporar los reportes/vistas del dominio de competencias (eventos, fases,
  disciplinas, pruebas, regiones, rankings) desde el data lake, con el mismo pipeline
  `preprocess_corpus.py` (separación de PII → gazetteer, enriquecimiento con ontología).
- **Ontología:** extender clases y sinónimos a competencias (Prueba Regional, Región, Fase,
  Disciplina, Evento) para que `φ(o_d)` cubra el nuevo dominio.
- **Corpus objetivo v2:** de 17 a **varios cientos** de documentos, lo que vuelve informativo
  el Recall@100 y da poder estadístico a las pruebas pareadas.
- **Validación:** repetir la escalera de ablación y el análisis por estratos; verificar que
  las consultas de competencias pasan de qrel vacío a tener objetivos reales.

## 4. (Punto 7) Sección lista para el LaTeX

> **Evaluación experimental.** Se evaluó una escalera de ablación de cinco configuraciones
> sobre un protocolo de recuperación por documento conocido (*known-item*), con el codificador
> multilingüe `jina-embeddings-v3` (congelado, solo inferencia) y BM25 con analizador en
> español. La métrica principal fue nDCG@10, complementada con MRR@10, Recall@100 y P@10,
> desglosadas por estrato de consulta. La configuración **híbrida con inyección de ontología**
> obtuvo el mejor desempeño global (nDCG@10 = 0.936) y el máximo en el estrato de entidad
> exacta, confirmando que la ventaja del recuperador híbrido se concentra en consultas por
> nombre propio (hipótesis central). La inyección de metadatos de la ontología mejoró de forma
> consistente tanto al recuperador denso como al híbrido. El realce suave por entidad no
> produjo mejoras y se descartó para la primera versión, al evidenciarse un detector de
> entidad de baja precisión. En eficiencia, la truncación Matryoshka del embedding permitió
> reducir la memoria del índice manteniendo una retención de nDCG@10 cercana a la unidad.
>
> **Limitaciones del protocolo.** El corpus de la versión 1 (17 reportes) y el uso de
> consultas *known-item* derivadas de los propios documentos implican que (i) Recall@100 es
> trivialmente 1.0 y (ii) el valor absoluto de las métricas no debe interpretarse como
> precisión en producción, sino como una comparación relativa, controlada y reproducible entre
> configuraciones. La validez externa se abordará en la versión 2 mediante un conjunto de
> consultas reales con juicios de expertos y la extensión del corpus al dominio de
> competencias.

*(Las tablas de §4 del notebook 01 y las figuras de los notebooks 01 y 03 se insertan aquí.)*

## 5. Próximos pasos (v1 → v2)

| Prioridad | Acción | Notebook / artefacto |
|---|---|---|
| Alta | Fijar `m` de Matryoshka con retención ≥ 0.99 y documentar índice/latencia | 03 |
| Alta | Conjunto de consultas reales con juicios de expertos (80–120) | plan §2 |
| Alta | Extender corpus a Competencias (cientos de docs) | plan §3, `preprocess_corpus.py` |
| Media | Detector de entidad de alta precisión → reactivar realce (V1/V2) | 02 |
| Media | Migrar índice denso a HNSW al crecer el corpus | despliegue |
| Baja | Explorar re-ranking (cross-encoder) como peldaño futuro | v2 |

**Cierre.** La v1 entrega una recomendación defendible y reproducible (**Híbrido + ontología,
sin realce, embedding truncado**), con las limitaciones del protocolo declaradas de forma
transparente. La v2 se centra en **validez externa** (consultas reales) y **cobertura**
(corpus de competencias), que son las dos palancas que hoy limitan la interpretación.